##Order Table


In [0]:
from pyspark.sql.functions import col,when,current_timestamp
from pyspark.sql.types import TimestampType

order_df=spark.read.format("delta").table("olist_project.bronze.orders")
order_df=order_df.select(col("order_approved_at").cast(TimestampType()).alias("order_approved_at"),
                         col("order_delivered_carrier_date").cast(TimestampType()).alias("order_delivered_carrier_date"),
                         col("order_delivered_customer_date").cast(TimestampType()).alias("order_delivered_customer_date"),
                         col("order_estimated_delivery_date").cast(TimestampType()).alias("order_estimated_delivery_date"),
                         col("order_id"),
                         col("customer_id"),
                         col("order_purchase_timestamp").cast(TimestampType()).alias("order_purchase_timestamp"),
                         col("order_status")).withColumn("processed_at", current_timestamp()).withColumn("is_approval_missing",when((col("order_approved_at").isNull())& (col("order_status")=="delivered"),1).otherwise(0))
        
order_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.silver.orders")

In [0]:
spark.sql("""
SELECT 
    COUNT(*) as total_rows,
    SUM(is_approval_missing) as anomaly_count,
    COUNT(CASE WHEN order_approved_at IS NULL THEN 1 END) as null_approvals
FROM olist_project.silver.orders
""").show()

##Customer Table

In [0]:
%sql
drop table olist_project.silver.order_items;

In [0]:
from pyspark.sql.functions import lower, upper,trim,current_timestamp

customer_df=spark.read.format("delta").table("olist_project.bronze.customers")
customer_df=customer_df.withColumn("customer_city",lower(trim(col("customer_city"))))\
                        .withColumn("customer_state",upper(trim(col("customer_state"))))\
                        .withColumn("processed_at", current_timestamp())

customer_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.silver.customers")


In [0]:
spark.sql("""
SELECT COUNT(*) as total_rows,
       COUNT(DISTINCT customer_city) as unique_cities
FROM olist_project.silver.customers
""").show()

##Payment


In [0]:
from pyspark.sql.types import DoubleType, IntegerType
import pyspark.sql.functions as F

payment_df=spark.read.format("delta").table("olist_project.bronze.payments")
payment_df=payment_df.select(F.col("order_id"),
                             F.col("payment_sequential").cast(IntegerType()),
                             F.col("payment_type"),
                             F.col("payment_installments").cast(IntegerType()),
                             F.col("payment_value").cast(DoubleType())).withColumn("processed_at", F.current_timestamp())

payment_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.silver.payments")


In [0]:
spark.sql("""
SELECT COUNT(*) as total_rows
FROM olist_project.silver.payments
""").show()

##Product

In [0]:
from pyspark.sql.functions import col,current_timestamp,when
product_df=spark.read.format("delta").table("olist_project.bronze.products")

product_df=product_df.select(
    col("product_id"),
    when(col("product_category_name").isNull(), "unknown").otherwise(col("product_category_name")).alias("product_category_name"),
    col("product_name_lenght").cast("double"),
    col("product_description_lenght").cast("double"),
    col("product_photos_qty").cast("integer"),
    col("product_weight_g").cast("double"),
    col("product_length_cm").cast("double"),
    col("product_height_cm").cast("double"),
    col("product_width_cm").cast("double")).withColumn("processed_at", current_timestamp())

product_df.write.format("delta").mode("overwrite").saveAsTable("olist_project.silver.products")

In [0]:
spark.sql("""
SELECT COUNT(*) as total_rows
FROM olist_project.silver.products
""").show()

##Reviews

In [0]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import trim
reviews_df=spark.read.format("delta").table("olist_project.bronze.reviews")
reviews_df=reviews_df.select(col("review_id"),
                             col("order_id"),
                             col("review_score").cast("integer"),
                             col("review_comment_title"),
                             col("review_comment_message"),
                             col("review_creation_date").cast(TimestampType()).alias("review_creation_date"),
                             col("review_answer_timestamp").cast(TimestampType()).alias("review_answer_timestamp")).withColumn("processed_at", current_timestamp()).withColumn("review_comment_message", trim(col("review_comment_message"))).withColumn("review_comment_title", trim(col("review_comment_title")))
        
reviews_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_project.silver.reviews")



In [0]:
spark.sql("""
SELECT COUNT(*) as total_rows
FROM olist_project.silver.reviews
""").show()

##Order Items

In [0]:
from pyspark.sql.functions import col, current_timestamp
order_items_df=spark.read.format("delta").table("olist_project.bronze.order_items")

order_items_df=order_items_df.select(
    col("order_id"),
    col("order_item_id").cast("integer"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date").cast("timestamp"),
    col("price").cast("double"),
    col("freight_value").cast("double")
).withColumn("processed_at",current_timestamp())
order_items_df.write.format("delta").mode("overwrite").saveAsTable("olist_project.silver.order_items")

In [0]:
spark.sql("""
SELECT COUNT(*) as total_rows
FROM olist_project.silver.order_items
""").show()

In [0]:
for table in ["orders", "customers", "order_items", "payments", "reviews", "products"]:
    bronze = spark.sql(f"SELECT COUNT(*) as cnt FROM olist_project.bronze.{table}").collect()[0][0]
    silver = spark.sql(f"SELECT COUNT(*) as cnt FROM olist_project.silver.{table}").collect()[0][0]
    print(f"{table}: bronze={bronze}, silver={silver}, match={bronze==silver}")

In [0]:
spark.sql("""
    SELECT 
        COUNT(*) as total_orders,
        COUNT(DISTINCT o.order_id) as unique_orders,
        COUNT(DISTINCT o.customer_id) as unique_customers,
        COUNT(DISTINCT c.customer_id) as customers_with_match
    FROM olist_project.silver.orders o
    LEFT JOIN olist_project.silver.customers c 
        ON o.customer_id = c.customer_id
""").show()